# Two-Stage Adaptive Conformal — Manheim Used-Car Index

Adaptive conformal prediction on the real Manheim used-vehicle price index,
using the two-stage forecasting pipeline (`two_stage.py`) and the shared
adaptive methods in `adaptive_conformal.py`.

1. Load the Manheim series and indicators.
2. Pipelines: `conformal_r1_r2` (two-stage), `conformal_single` (single-stage).
3. Main two-stage experiment (α=0.2): width / coverage / interval plots.
4. The learned `a`, `b` scaling weights.
5. Single-stage ablations (VAR, ARIMA): same comparison but the baselines
   run on a single-stage forecaster (performance is similar, the model is not the reason for baseline performance)
6. Coverage / width as α varies (run last — these overwrite `res`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

from utils import *
from two_stage import *
from adaptive_conformal import (
    adaptive_fwer_alpha, aci, aci_clipped, quantile_integrator_log_scorecaster,
    decay_quantile, dtaci, ECI, sliding_window_average,
)

## 1. Load Manheim data and indicators

In [ ]:
indicator_df = get_indicators("Indicators")
indicator_df = indicator_df.resample('MS').first().interpolate()

In [ ]:
manheim_df = indicator_df[['manheim']]
indicator_df.drop(['manheim'],axis = 1)
manheim_df = manheim_df.dropna()

In [ ]:
data = indicator_df, manheim_df
test_date = '06-2010'
test_end = '06-2023'
test_last_index = manheim_df['manheim'].index.get_loc(pd.to_datetime(test_end))
test_init_index = manheim_df['manheim'].index.get_loc(pd.to_datetime(test_date))

## 2. Conformal pipelines (two-stage and single-stage)

In [ ]:
def conformal_r1_r2(indicators, upstream_models, downstream, data,
                    window_length, lambda_set, conformal_date, test_date, test_end, T_burnin, initializations, horizon,
                    alpha, delta, lr, alpha_lr, epsilon):

    conformal_r1_score = []
    conformal_r2_score = []
    conformal_scores = []
    indicator_df,manheim_df = data
    forecasts = []
    truths = []

    for unseen_date in tqdm(pd.date_range(conformal_date, pd.to_datetime(test_end), freq='MS')):
        upstream = CombinedUpstream(models=upstream_models, test_date=unseen_date, data = data)

        model = TwoStageModel(indicators, upstream, downstream, unseen_date, data)

        forecast = model.forecast(horizon)

        perfect_up_forecast = model.perfect_upstream_forecast(horizon)

        truth = manheim_df['manheim'].loc[pd.to_datetime(unseen_date)]
        if(unseen_date >=pd.to_datetime(test_date)):
            forecasts.append(forecast)
            truths.append(truth)

        conformal_score = np.abs(truth - forecast)
        r2 = np.abs(truth - perfect_up_forecast)
        r1 = np.abs(conformal_score - r2)
        conformal_r1_score.append(r1)
        conformal_r2_score.append(r2)
        conformal_scores.append(conformal_score)

    conformal_r1_score = np.asarray(conformal_r1_score)
    conformal_r2_score = np.asarray(conformal_r2_score)
    conformal_scores = np.asarray(conformal_scores)

    res = adaptive_fwer_alpha(
        conformal_scores,
        alpha,
        epsilon,
        window_length,
        window_length,
        T_burnin,
        lambda_set,
        conformal_r1_score,
        conformal_r2_score,
        delta,
        initializations,
        horizon,
        lr,
        alpha_lr)

    res_aci = aci(conformal_scores,
    alpha,
    alpha_lr,
    window_length,
    T_burnin,
    horizon)
    steps = [0.001, 0.002,0.004, 0.008, 0.0160, 0.032, 0.064, 0.128]
    res_dtaci = dtaci(conformal_scores, 1-alpha, steps, ahead=horizon, window_size = window_length)

    train_size = 132

    Csat = 2/np.pi*(np.ceil(np.log(train_size*2 -T_burnin)*0.1) - 1/np.log(train_size*2 -T_burnin))
    KI = 20
    res_pid = quantile_integrator_log_scorecaster(conformal_scores,
    alpha,
    alpha_lr,
    indicator_df,
    T_burnin,
    Csat,
    KI,
    'upper',
    horizon,Integrate = True, scorecast = True, index_forecast = True)

    eps = 0.1
    etas_decaying = np.array([1000/(t**(1/2+eps)) for t in range(1, len(conformal_scores)+1)])
    q_1 = conformal_scores[0]
    res_decay = decay_quantile(conformal_scores, q_1, etas_decaying, alpha, horizon)

    res_eci = ECI(
                conformal_scores,
                alpha,
                alpha_lr,
                T_burnin,
                horizon
                )

    return res, res_aci, res_pid, res_decay, res_dtaci, res_eci, conformal_scores, np.asarray(forecasts), np.asarray(truths)

def conformal_r1_r2_us(indicators, upstream_models, downstream, data,
                    window_length, lambda_set, conformal_date, test_date, test_end, T_burnin, initializations, horizon,
                    alpha, delta, lr, alpha_lr, epsilon):

    conformal_r1_score = []
    conformal_r2_score = []
    conformal_scores = []
    indicator_df,manheim_df = data
    forecasts = []
    truths = []

    for unseen_date in tqdm(pd.date_range(conformal_date, pd.to_datetime(test_end), freq='MS')):
        upstream = CombinedUpstream(models=upstream_models, test_date=unseen_date, data = data)

        model = TwoStageModel(indicators, upstream, downstream, unseen_date, data)

        forecast = model.forecast(horizon)

        perfect_up_forecast = model.perfect_upstream_forecast(horizon)

        truth = manheim_df['manheim'].loc[pd.to_datetime(unseen_date)]
        if(unseen_date >=pd.to_datetime(test_date)):
            forecasts.append(forecast)
            truths.append(truth)

        conformal_score = np.abs(truth - forecast)
        r2 = np.abs(truth - perfect_up_forecast)
        r1 = np.abs(conformal_score - r2)
        conformal_r1_score.append(r1)
        conformal_r2_score.append(r2)
        conformal_scores.append(conformal_score)

    conformal_r1_score = np.asarray(conformal_r1_score)
    conformal_r2_score = np.asarray(conformal_r2_score)
    conformal_scores = np.asarray(conformal_scores)

    res = adaptive_fwer_alpha(
        conformal_scores,
        alpha,
        epsilon,
        window_length,
        window_length,
        T_burnin,
        lambda_set,
        conformal_r1_score,
        conformal_r2_score,
        delta,
        initializations,
        horizon,
        lr,
        alpha_lr)

    return res, conformal_scores, np.asarray(forecasts), np.asarray(truths)

In [ ]:
def conformal_single(model_dict, data,
                     window_length, lambda_set, conformal_date, test_date, test_end,
                     T_burnin, initializations, horizon,
                     alpha, delta, lr, alpha_lr, epsilon):
    """Single-stage baseline pipeline on the Manheim series.

    Rolls a single forecaster (VAR or ARIMA, per model_dict['model_class'])
    forward and returns the adaptive baselines (ACI, PID, OCID, DtACI, ECI)
    on its conformal scores. There is no split residual here — the proposed
    two-stage FWER method (`res`) comes from the two-stage experiment above.
    """
    indicator_df, manheim_df = data
    conformal_scores, forecasts, truths = [], [], []
    for unseen_date in tqdm(pd.date_range(conformal_date, pd.to_datetime(test_end), freq='MS')):
        kw = model_dict['kwargs']
        if model_dict['model_class'] is IndicatorVAR:
            model = IndicatorVAR(model_dict['indicators'], kw['lags'], kw['trend'],
                                 unseen_date, data, output_indicators=['manheim'],
                                 transform=kw['transform'], horizon=6)
        else:
            model = IndicatorARIMA(model_dict['indicators'], kw['order'], kw['trend'],
                                   unseen_date, data, transform=kw['transform'], horizon=6)
        forecast = model.get_forecast(horizon)['manheim'].iloc[0]
        truth = manheim_df['manheim'].loc[pd.to_datetime(unseen_date)]
        if unseen_date >= pd.to_datetime(test_date):
            forecasts.append(forecast)
            truths.append(truth)
        conformal_scores.append(np.abs(truth - forecast))

    conformal_scores = np.asarray(conformal_scores)

    res_aci = aci_clipped(conformal_scores, alpha, alpha_lr, window_length, T_burnin, horizon)

    train_size = 132
    Csat = 2 / np.pi * (np.ceil(np.log(train_size * 2 - T_burnin) * 0.1) - 1 / np.log(train_size * 2 - T_burnin))
    KI = 20
    res_pid = quantile_integrator_log_scorecaster(
        conformal_scores, alpha, alpha_lr, indicator_df, T_burnin, Csat, KI,
        'upper', horizon, Integrate=True, scorecast=True, index_forecast = True,
    )

    eps = 0.1
    etas_decaying = np.array([1000 / (t ** (1 / 2 + eps)) for t in range(1, len(conformal_scores) + 1)])
    res_decay = decay_quantile(conformal_scores, conformal_scores[0], etas_decaying, alpha, horizon)

    steps = [0.001, 0.002, 0.004, 0.008, 0.0160, 0.032, 0.064, 0.128]
    res_dtaci = dtaci(conformal_scores, alpha, steps, ahead=horizon, window_size=window_length)

    res_eci = ECI(conformal_scores, alpha, alpha_lr, T_burnin, horizon)

    return res_aci, res_pid, res_decay, res_dtaci, res_eci, conformal_scores, np.asarray(forecasts), np.asarray(truths)

## 3. Main two-stage experiment (α = 0.2)

Obtains resuls (`res` (our Split Residual with FWER method)), and the baselines.

In [ ]:
horizon = 6
sliding_window_len =40
test_date = '06-2010'
test_end = '06-2023'
conformal_date = '06-2008'
VAR_indicators = ['WPU101707', 'CUSR0000SETA02', 'PCU33443344', 'AUINSA'] # steel PPI, used auto CPI, semiconductor PPI, auto inventory
ARIMA_indicators = ['PCECC96']
models = [{'indicators': VAR_indicators, 'model_class': IndicatorVAR, 'kwargs': {'lags': 3, 'trend': 'ctt', 'transform': 'log-differencing'}},
          {'indicators': 'PCECC96', 'model_class': IndicatorARIMA, 'kwargs': {'order': (2,0,0), 'trend': 'ct', 'transform': None}}]
indicators = VAR_indicators + ARIMA_indicators

alpha=0.2
epsilon=0.01
delta=0.101
lr=0.01

grid = np.linspace(0.0, 1, 8)
lambda_a,lambda_b = np.meshgrid(grid,grid)
lambda_set = np.column_stack((lambda_a.flatten(), lambda_b.flatten()))

alpha_lrs = [0.01]

In [ ]:
for i in range(len(alpha_lrs)):
    alpha_lr = alpha_lrs[i]
    window_length = 36
    T_burnin= 24
    res,res_aci,res_pid, res_decay,res_dtaci,res_eci, conformal_scores,preds, truths = conformal_r1_r2(indicators, models, 'ARIMA', data,
                        window_length, lambda_set, conformal_date, test_date, test_end, T_burnin, None, horizon,
                        alpha, delta, lr, alpha_lr, epsilon)

    plt.plot(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),np.abs(res['q'][window_length+T_burnin:]), label="Split Residuals", linestyle="-", color="C0")
    plt.plot(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),np.abs(res_aci['q'][window_length+T_burnin:]), label="ACI (clipped)", linestyle="--", color="C1")
    plt.plot(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),np.abs(res_pid['q'][window_length+T_burnin:]), label="PID", linestyle="-.", color="C2")
    plt.plot(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),np.abs(res_decay['q'][window_length+T_burnin:]), label="OCID", linestyle="-.", color="C3")
    plt.plot(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),np.abs(res_eci['q'][window_length+T_burnin:]), label="ECI", linestyle="--", color="C4")
    plt.plot(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),np.abs(res_dtaci['q'][window_length+T_burnin:]), label="DtACI (clipped)", linestyle="--", color="C5")
    plt.xlabel("Date")

    plt.ylabel("Prediction Interval Width")
    plt.legend(loc="lower right", frameon=False, fontsize=8)
    plt.tight_layout()
    plt.savefig(f"pi_lm_width_{i}_weci.png", dpi=300, bbox_inches='tight')

    plt.show()

    array_cov_us = sliding_window_average(res['cov'], sliding_window_len)
    plt.plot(manheim_df.index[test_last_index-116:test_last_index].to_numpy(),array_cov_us[T_burnin+window_length +horizon- sliding_window_len:],linestyle="-", color = 'C0', label= 'Split Residuals')

    array_cov_aci = sliding_window_average(res_aci['cov'], sliding_window_len)
    plt.plot(manheim_df.index[test_last_index-116:test_last_index].to_numpy(),array_cov_aci[T_burnin+window_length +horizon- sliding_window_len:],linestyle="--", color = 'C1', label = 'ACI (clipped)')

    array_cov_eci = sliding_window_average(res_eci['cov'], sliding_window_len)
    plt.plot(manheim_df.index[test_last_index-116:test_last_index].to_numpy(),array_cov_eci[T_burnin+window_length +horizon- sliding_window_len:],linestyle="--", color = 'C4', label = 'ECI')

    array_cov_dtaci = sliding_window_average(res_dtaci['cov'], sliding_window_len)
    plt.plot(manheim_df.index[test_last_index-116:test_last_index].to_numpy(),array_cov_dtaci[T_burnin+window_length +horizon- sliding_window_len:],linestyle="--", color = 'C5', label = 'DtACI (clipped)')

    array_cov_pid = sliding_window_average(res_pid['cov'], sliding_window_len)
    plt.plot(manheim_df.index[test_last_index-116:test_last_index].to_numpy(),array_cov_pid[T_burnin+window_length +horizon- sliding_window_len:], linestyle="-.",color = 'C2', label='PID')

    array_cov_decay = sliding_window_average(res_decay['cov'], sliding_window_len)
    plt.plot(manheim_df.index[test_last_index-116:test_last_index].to_numpy(),array_cov_decay[T_burnin+window_length +horizon- sliding_window_len:], linestyle="-.",color = 'C3', label='OCID')

    plt.axhline(0.8, color='gray', linestyle='--', linewidth=1, label='80% Coverage')
    plt.xlabel("Date")

    plt.ylabel("Average Coverage (40 Sliding Window)")
    plt.ylim(0.45, 1.01)
    plt.legend(loc="upper right", frameon=False, fontsize=8)
    plt.savefig(f"pi_lm_cov_{i}_weci.png", dpi=300, bbox_inches='tight')
    plt.show()

    figsize = (8, 4)
    plt.figure(figsize=figsize)

    plt.fill_between(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),preds[window_length:]- res['q'][window_length+T_burnin:],preds[window_length:]+ res['q'][window_length+T_burnin:], label="Split Residual Interval", color="C0",alpha=0.2)
    plt.plot(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),preds[window_length:], label ='Predicted Value', color = 'C6')
    plt.plot(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),truths[window_length:], label = 'True Value', color = 'C7')
    plt.xlabel("Date")
    plt.ylabel("Manheim Index")
    plt.ylim(0, 370)
    plt.legend(loc="upper left", frameon=False, fontsize=10)

    plt.savefig(f"us_actual_lm_{i}.png", dpi=300, bbox_inches='tight')
    plt.show()
    plt.figure(figsize=figsize)
    plt.fill_between(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),preds[window_length:]- res_aci['q'][window_length+T_burnin:],preds[window_length:]+ res_aci['q'][window_length+T_burnin:], label="ACI (Clipped) Interval", color="C1",alpha=0.2)
    plt.plot(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),preds[window_length:], label ='Predicted Value', color = 'C6')
    plt.plot(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),truths[window_length:], label = 'True Value', color = 'C7')
    plt.xlabel("Date")
    plt.ylabel("Manheim Index")
    plt.ylim(0, 370)
    plt.legend(loc="upper left", frameon=False, fontsize=10)

    plt.savefig(f"aci_actual_lm_{i}.png", dpi=300, bbox_inches='tight')
    plt.show()

    plt.figure(figsize=figsize)

    plt.fill_between(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),preds[window_length:]- np.abs(res_pid['q'][window_length+T_burnin:]),preds[window_length:]+ np.abs(res_pid['q'][window_length+T_burnin:]), label="PID Interval", color="C2",alpha=0.2)
    plt.plot(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),preds[window_length:], label ='Predicted Value', color = 'C6')
    plt.plot(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),truths[window_length:], label = 'True Value', color = 'C7')
    plt.xlabel("Date")
    plt.ylabel("Manheim Index")
    plt.ylim(0, 370)
    plt.legend(loc="upper left", frameon=False, fontsize=10)

    plt.savefig(f"pid_actual_lm_{i}.png", dpi=300, bbox_inches='tight')
    plt.show()
    plt.figure(figsize=figsize)

    plt.fill_between(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),preds[window_length:]- res_decay['q'][window_length+T_burnin:],preds[window_length:]+ res_decay['q'][window_length+T_burnin:], label="OCID Interval", color="C3",alpha=0.2)
    plt.plot(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),preds[window_length:], label ='Predicted Value', color = 'C6')
    plt.plot(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),truths[window_length:], label = 'True Value', color = 'C7')
    plt.xlabel("Date")
    plt.ylabel("Manheim Index")
    plt.ylim(0, 370)
    plt.legend(loc="upper left", frameon=False, fontsize=10)

    plt.savefig(f"ocid_actual_lm_{i}.png", dpi=300, bbox_inches='tight')
    plt.show()

    plt.figure(figsize=figsize)
    plt.fill_between(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),preds[window_length:]- res_dtaci['q'][window_length+T_burnin:],preds[window_length:]+ res_dtaci['q'][window_length+T_burnin:], label="DtACI Interval", color="C5",alpha=0.2)
    plt.plot(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),preds[window_length:], label ='Predicted Value', color = 'C5')
    plt.plot(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),truths[window_length:], label = 'True Value', color = 'C6')
    plt.xlabel("Date")
    plt.ylabel("Manheim Index")
    plt.ylim(0, 370)
    plt.legend(loc="upper left", frameon=False, fontsize=10)

    plt.savefig(f"dtaci_actual_lm_{i}.png", dpi=300, bbox_inches='tight')
    plt.show()

    plt.figure(figsize=figsize)
    plt.fill_between(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),preds[window_length:]- res_eci['q'][window_length+T_burnin:],preds[window_length:]+ res_eci['q'][window_length+T_burnin:], label="ECI", color="C4",alpha=0.2)
    plt.plot(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),preds[window_length:], label ='Predicted Value', color = 'C6')
    plt.plot(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),truths[window_length:], label = 'True Value', color = 'C7')
    plt.xlabel("Date")
    plt.ylabel("Manheim Index")
    plt.ylim(0, 370)
    plt.legend(loc="upper left", frameon=False, fontsize=10)

    plt.savefig(f"eci_actual_lm_{i}.png", dpi=300, bbox_inches='tight')
    plt.show()

## 4. Learned a/b scaling weights → `abplotlm.png`

In [ ]:
plt.plot(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),res['a'][window_length+T_burnin:], color = 'firebrick', label = r'$a$ parameter' )
plt.plot(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),res['b'][window_length+T_burnin:], color = 'rebeccapurple', label = r'$b$ parameter')
plt.xlabel("Timestep")
plt.ylabel('Scaling Value')
plt.legend(loc="lower right", frameon=False, fontsize=8)
plt.savefig("ab_plot.png", dpi=300, bbox_inches='tight')

## 5. Coverage and width across α

Sweeps α over 0.1–0.9 

In [ ]:
alphas=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]
epsilon=0.01
delta=0.5
lr=0.01

grid = np.linspace(0.0, 1, 12)
lambda_a,lambda_b = np.meshgrid(grid,grid)
lambda_set = np.column_stack((lambda_a.flatten(), lambda_b.flatten()))

res_ac = []
res_pc = []
res_dc = []
res_dtc = []
res_ec = []

res_aw = []
res_pw = []
res_dw = []
res_dtw = []
res_ew = []

alpha_lr = 0.01
for i in range(len(alphas)):
    alpha = alphas[i]

    window_length = 36
    T_burnin= 24
    res,res_aci,res_pid, res_decay,res_dtaci,res_eci, conformal_scores,preds, truths = conformal_r1_r2(indicators, models, 'ARIMA', data,
                        window_length, lambda_set, conformal_date, test_date, test_end, T_burnin, None, horizon,
                        alpha, delta, lr, alpha_lr, epsilon)

    res_ac.append(np.mean(res_aci['cov'][-50:]))
    res_pc.append(np.mean(res_pid['cov'][-50:]))
    res_dc.append(np.mean(res_decay['cov'][-50:]))
    res_dtc.append(np.mean(res_dtaci['cov'][-50:]))
    res_ec.append(np.mean(res_eci['cov'][-50:]))

    res_aw.append(np.mean(res_aci['q'][-50:]))
    res_pw.append(np.mean(res_pid['q'][-50:]))
    res_dw.append(np.mean(res_decay['q'][-50:]))
    res_dtw.append(np.mean(res_dtaci['q'][-50:]))
    res_ew.append(np.mean(res_eci['q'][-50:]))

In [ ]:
alphas=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]
epsilon=0.01
delta=0.3
lr=0.01
epsilons=[0.001, 0.05, 0.1, 0.28,0.38,0.47,0.7, 0.8, 0.9]
grid = np.linspace(0.1, 1, 10)
res_c=[]
res_w=[]

alpha_lr = 0.01
for i in range(len(alphas)):

    epsilon = epsilons[i]
    lambda_a,lambda_b = np.meshgrid(grid,grid)
    lambda_set = np.column_stack((lambda_a.flatten(), lambda_b.flatten()))
    alpha = alphas[i]

    window_length = 36
    T_burnin= 24
    res,conformal_scores,preds, truths = conformal_r1_r2_us(indicators, models, 'ARIMA', data,
                        window_length, lambda_set, conformal_date, test_date, test_end, T_burnin, None, horizon,
                        alpha, delta, lr, alpha_lr, epsilon)
    finite_mask = np.isfinite(res['q'][-50:])
    res_c.append(np.mean(res['cov'][-50:][finite_mask]))
    vals = res['q'][-50:]
    res_w.append(np.mean(vals[np.isfinite(vals)]))

In [ ]:
alpha_inv = [1- val for val in alphas]
alpha_inv.reverse()

plt.plot(alpha_inv, res_c[::-1],color="C0",label="Split Residuals", linestyle="-")
plt.plot(alpha_inv, res_ac[::-1],color="C1",label="ACI (clipped)", linestyle="--")
plt.plot(alpha_inv, res_pc[::-1],color="C2", label="PID", linestyle="-.")
plt.plot(alpha_inv, res_dc[::-1],color="C3", label="OCID", linestyle="-.")
plt.plot(alpha_inv, res_ec[::-1],color="C4",label="ECI", linestyle="--")
plt.plot(alpha_inv,alpha_inv, color = 'black',label='Perfect Calibration')
plt.xlabel("Desired Coverage Level")
plt.ylabel("Average Empirical Coverage")
plt.legend()
plt.savefig("coverage_v_coverage.png", dpi=300)



In [ ]:
plt.plot(alpha_inv, res_w[::-1],color="C0",label="Split Residuals", linestyle="-")
plt.plot(alpha_inv, res_aw[::-1],color="C1",label="ACI (clipped)", linestyle="--")
plt.plot(alpha_inv, res_pw[::-1],color="C2", label="PID", linestyle="-.")
plt.plot(alpha_inv, np.abs(res_dw[::-1]),color="C3", label="OCID", linestyle="-.")
plt.plot(alpha_inv, res_ew[::-1],color="C4",label="ECI", linestyle="--")
plt.xlabel("Desired Coverage Level")
plt.ylabel("Average Width")
plt.legend()
plt.savefig("coverage_v_width.png", dpi=300)

## 6. Single-stage ablation — VAR

`conformal_single` with the VAR forecaster. 'Split Residuals' is the two-stage
`res`, ACI/PID/OCID/ECI/DtACI are the single-stage baselines.


In [ ]:
horizon = 6
sliding_window_len =40
test_date = '06-2010'
test_end = '06-2023'
conformal_date = '06-2008'
VAR_indicators = ['WPU101707', 'CUSR0000SETA02', 'PCU33443344', 'AUINSA','PCECC96','manheim']
ARIMA_indicators = ['PCECC96']

model_dict = {'indicators': VAR_indicators, 'model_class': IndicatorVAR, 'kwargs': {'lags': 3, 'trend': 'ctt', 'transform': 'log-differencing'}}
indicators = VAR_indicators + ARIMA_indicators

alpha=0.2
epsilon=0.01
delta=0.9
lr=0.01

grid = np.linspace(0.0, 1, 8)
lambda_a,lambda_b = np.meshgrid(grid,grid)
lambda_set = np.column_stack((lambda_a.flatten(), lambda_b.flatten()))

alphas = [0.01]
for i in range(len(alphas)):
    alpha_lr = alphas[i]

    window_length = 36
    T_burnin= 24
    res_aci,res_pid, res_decay, res_dtaci, res_eci, conformal_scores,preds, truths = conformal_single(model_dict, data,
                        window_length, lambda_set, conformal_date, test_date, test_end, T_burnin, None, horizon,
                        alpha, delta, lr, alpha_lr, epsilon)

    plt.plot(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),res['q'][window_length+T_burnin:], label="Split Residuals", linestyle="-", color="C0")
    plt.plot(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),res_aci['q'][window_length+T_burnin:], label="ACI (clipped)", linestyle="--", color="C1")
    plt.plot(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),np.abs(res_pid['q'][window_length+T_burnin:]), label="PID", linestyle="-.", color="C2")
    plt.plot(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),np.abs(res_decay['q'][window_length+T_burnin:]), label="OCID", linestyle="-.", color="C3")
    plt.plot(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),res_eci['q'][window_length+T_burnin:], label="ECI", linestyle="--", color="C4")
    plt.plot(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),res_dtaci['q'][window_length+T_burnin:], label="DtACI", linestyle="--", color="C5")
    plt.xlabel("Date")

    plt.ylabel("Prediction Interval Width")
    plt.legend(loc="lower right", frameon=False, fontsize=8)
    plt.tight_layout()
    plt.savefig(f"var_lm_width_{i}.png", dpi=300, bbox_inches='tight')

    plt.show()

    array_cov_us = sliding_window_average(res['cov'], sliding_window_len)
    plt.plot(manheim_df.index[test_last_index-116:test_last_index].to_numpy(),array_cov_us[T_burnin+window_length +horizon- sliding_window_len:],linestyle="-", color = 'C0', label= 'Split Residuals')

    array_cov_aci = sliding_window_average(res_aci['cov'], sliding_window_len)
    plt.plot(manheim_df.index[test_last_index-116:test_last_index].to_numpy(),array_cov_aci[T_burnin+window_length +horizon- sliding_window_len:],linestyle="--", color = 'C1', label = 'ACI (clipped)')

    array_cov_pid = sliding_window_average(res_pid['cov'], sliding_window_len)
    plt.plot(manheim_df.index[test_last_index-116:test_last_index].to_numpy(),array_cov_pid[T_burnin+window_length +horizon- sliding_window_len:], linestyle="-.",color = 'C2', label='PID')

    array_cov_decay = sliding_window_average(res_decay['cov'], sliding_window_len)
    plt.plot(manheim_df.index[test_last_index-116:test_last_index].to_numpy(),array_cov_decay[T_burnin+window_length +horizon- sliding_window_len:], linestyle="-.",color = 'C3', label='OCID')

    array_cov_eci = sliding_window_average(res_eci['cov'], sliding_window_len)
    plt.plot(manheim_df.index[test_last_index-116:test_last_index].to_numpy(),array_cov_eci[T_burnin+window_length +horizon- sliding_window_len:], linestyle="-.",color = 'C4', label='ECI')

    array_cov_dtaci = sliding_window_average(res_dtaci['cov'], sliding_window_len)
    plt.plot(manheim_df.index[test_last_index-116:test_last_index].to_numpy(),array_cov_dtaci[T_burnin+window_length +horizon- sliding_window_len:], linestyle="-.",color = 'C5', label='DtACI')

    plt.axhline(0.8, color='gray', linestyle='--', linewidth=1, label='80% Coverage')
    plt.xlabel("Date")

    plt.ylabel("Average Coverage (40 Sliding Window)")
    plt.ylim(0.45, 1.01)
    plt.legend(loc="upper right", frameon=False, fontsize=8)
    plt.savefig(f"var_lm_cov_{i}.png", dpi=300, bbox_inches='tight')
    plt.show()

    plt.fill_between(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),preds[window_length:]- res_aci['q'][window_length+T_burnin:],preds[window_length:]+ res_aci['q'][window_length+T_burnin:], label="ACI (Clipped) Interval", color="C1",alpha=0.2)
    plt.plot(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),preds[window_length:], label ='Predicted Value', color = 'C6')
    plt.plot(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),truths[window_length:], label = 'True Value', color = 'C5')
    plt.xlabel("Date")
    plt.ylabel("Manheim Index")
    plt.ylim(0, 370)
    plt.legend(loc="upper left", frameon=False, fontsize=10)

    plt.savefig(f"var_aci_lm_{i}.png", dpi=300, bbox_inches='tight')
    plt.show()
    plt.fill_between(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),preds[window_length:]- res_pid['q'][window_length+T_burnin:],preds[window_length:]+ res_pid['q'][window_length+T_burnin:], label="PID Interval", color="C2",alpha=0.2)
    plt.plot(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),preds[window_length:], label ='Predicted Value', color = 'C6')
    plt.plot(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),truths[window_length:], label = 'True Value', color = 'C5')
    plt.xlabel("Date")
    plt.ylabel("Manheim Index")
    plt.ylim(0, 370)
    plt.legend(loc="upper left", frameon=False, fontsize=10)

    plt.savefig(f"var_pid_lm_{i}.png", dpi=300, bbox_inches='tight')
    plt.show()

    plt.fill_between(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),preds[window_length:]- res_decay['q'][window_length+T_burnin:],preds[window_length:]+ res_decay['q'][window_length+T_burnin:], label="OCID Interval", color="C3",alpha=0.2)
    plt.plot(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),preds[window_length:], label ='Predicted Value', color = 'C6')
    plt.plot(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),truths[window_length:], label = 'True Value', color = 'C5')
    plt.xlabel("Date")
    plt.ylabel("Manheim Index")
    plt.ylim(0, 370)
    plt.legend(loc="upper left", frameon=False, fontsize=10)

    plt.savefig(f"var_ocid_lm_{i}.png", dpi=300, bbox_inches='tight')
    plt.show()

    plt.fill_between(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),preds[window_length:]- res_eci['q'][window_length+T_burnin:],preds[window_length:]+ res_eci['q'][window_length+T_burnin:], label="ECI Interval", color="C4",alpha=0.2)
    plt.plot(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),preds[window_length:], label ='Predicted Value', color = 'C6')
    plt.plot(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),truths[window_length:], label = 'True Value', color = 'C5')
    plt.xlabel("Date")
    plt.ylabel("Manheim Index")
    plt.ylim(0, 370)
    plt.legend(loc="upper left", frameon=False, fontsize=10)

    plt.savefig(f"var_eci_lm_{i}.png", dpi=300, bbox_inches='tight')
    plt.show()

    plt.fill_between(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),preds[window_length:]- res_dtaci['q'][window_length+T_burnin:],preds[window_length:]+ res_dtaci['q'][window_length+T_burnin:], label="DtACI Interval", color="C5",alpha=0.2)
    plt.plot(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),preds[window_length:], label ='Predicted Value', color = 'C6')
    plt.plot(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),truths[window_length:], label = 'True Value', color = 'C5')
    plt.xlabel("Date")
    plt.ylabel("Manheim Index")
    plt.ylim(0, 370)
    plt.legend(loc="upper left", frameon=False, fontsize=10)

    plt.savefig(f"var_dtaci_lm_{i}.png", dpi=300, bbox_inches='tight')
    plt.show()

## 7. Single-stage ablation — ARIMA

As above with the ARIMA forecaster.

In [ ]:
horizon = 6
sliding_window_len =40
test_date = '06-2010'
test_end = '06-2023'
conformal_date = '06-2008'
VAR_indicators = ['WPU101707', 'CUSR0000SETA02', 'PCU33443344', 'AUINSA','PCECC96','manheim'] # steel PPI, used auto CPI, semiconductor PPI, auto inventory
ARIMA_indicators = ['PCECC96']

model_dict = {'indicators': 'manheim', 'model_class':IndicatorARIMA,'kwargs': {'order': (3,0,3), 'trend': 'ct', 'transform': None}}
indicators = VAR_indicators + ARIMA_indicators

alpha=0.2
epsilon=0.01
delta=0.9
lr=0.01

grid = np.linspace(0.0, 1, 8)
lambda_a,lambda_b = np.meshgrid(grid,grid)
lambda_set = np.column_stack((lambda_a.flatten(), lambda_b.flatten()))

alphas = [0.01]
for i in range(len(alphas)):
    alpha_lr = alphas[i]

    window_length = 36
    T_burnin= 24
    res_aci,res_pid, res_decay, res_dtaci, res_eci, conformal_scores,preds, truths = conformal_single(model_dict, data,
                        window_length, lambda_set, conformal_date, test_date, test_end, T_burnin, None, horizon,
                        alpha, delta, lr, alpha_lr, epsilon)

    plt.plot(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),res['q'][window_length+T_burnin:], label="Split Residuals", linestyle="-", color="C0")
    plt.plot(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),res_aci['q'][window_length+T_burnin:], label="ACI (clipped)", linestyle="--", color="C1")
    plt.plot(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),np.abs(res_pid['q'][window_length+T_burnin:]), label="PID", linestyle="-.", color="C2")
    plt.plot(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),np.abs(res_decay['q'][window_length+T_burnin:]), label="OCID", linestyle="-.", color="C3")
    plt.plot(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),np.abs(res_eci['q'][window_length+T_burnin:]), label="ECI", linestyle="-.", color="C4")
    plt.plot(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),np.abs(res_dtaci['q'][window_length+T_burnin:]), label="DtACI", linestyle="-.", color="C5")
    plt.xlabel("Date")

    plt.ylabel("Prediction Interval Width")
    plt.legend(loc="lower right", frameon=False, fontsize=8)
    plt.tight_layout()
    plt.savefig(f"arima_lm_width_{i}.png", dpi=300, bbox_inches='tight')

    plt.show()

    array_cov_us = sliding_window_average(res['cov'], sliding_window_len)
    plt.plot(manheim_df.index[test_last_index-116:test_last_index].to_numpy(),array_cov_us[T_burnin+window_length +horizon- sliding_window_len:],linestyle="-", color = 'C0', label= 'Split Residuals')

    array_cov_aci = sliding_window_average(res_aci['cov'], sliding_window_len)
    plt.plot(manheim_df.index[test_last_index-116:test_last_index].to_numpy(),array_cov_aci[T_burnin+window_length +horizon- sliding_window_len:],linestyle="--", color = 'C1', label = 'ACI (clipped)')

    array_cov_pid = sliding_window_average(res_pid['cov'], sliding_window_len)
    plt.plot(manheim_df.index[test_last_index-116:test_last_index].to_numpy(),array_cov_pid[T_burnin+window_length +horizon- sliding_window_len:], linestyle="-.",color = 'C2', label='PID')

    array_cov_decay = sliding_window_average(res_decay['cov'], sliding_window_len)
    plt.plot(manheim_df.index[test_last_index-116:test_last_index].to_numpy(),array_cov_decay[T_burnin+window_length +horizon- sliding_window_len:], linestyle="-.",color = 'C3', label='OCID')

    array_cov_eci = sliding_window_average(res_eci['cov'], sliding_window_len)
    plt.plot(manheim_df.index[test_last_index-116:test_last_index].to_numpy(),array_cov_eci[T_burnin+window_length +horizon- sliding_window_len:], linestyle="-.",color = 'C4', label='ECI')

    array_cov_dtaci = sliding_window_average(res_dtaci['cov'], sliding_window_len)
    plt.plot(manheim_df.index[test_last_index-116:test_last_index].to_numpy(),array_cov_dtaci[T_burnin+window_length +horizon- sliding_window_len:], linestyle="-.",color = 'C5', label='DtACI')

    plt.axhline(0.8, color='gray', linestyle='--', linewidth=1, label='80% Coverage')
    plt.xlabel("Date")

    plt.ylabel("Average Coverage (40 Sliding Window)")
    plt.ylim(0.45, 1.01)
    plt.legend(loc="upper right", frameon=False, fontsize=8)
    plt.savefig(f"arima_lm_cov_{i}.png", dpi=300, bbox_inches='tight')
    plt.show()

    plt.fill_between(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),preds[window_length:]- res_aci['q'][window_length+T_burnin:],preds[window_length:]+ res_aci['q'][window_length+T_burnin:], label="ACI (Clipped) Interval", color="C1",alpha=0.2)
    plt.plot(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),preds[window_length:], label ='Predicted Value', color = 'C6')
    plt.plot(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),truths[window_length:], label = 'True Value', color = 'C5')
    plt.xlabel("Date")
    plt.ylabel("Manheim Index")
    plt.ylim(0, 370)
    plt.legend(loc="upper left", frameon=False, fontsize=10)

    plt.savefig(f"arima_aci_lm_{i}.png", dpi=300, bbox_inches='tight')
    plt.show()
    plt.fill_between(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),preds[window_length:]- res_pid['q'][window_length+T_burnin:],preds[window_length:]+ res_pid['q'][window_length+T_burnin:], label="PID Interval", color="C2",alpha=0.2)
    plt.plot(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),preds[window_length:], label ='Predicted Value', color = 'C6')
    plt.plot(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),truths[window_length:], label = 'True Value', color = 'C5')
    plt.xlabel("Date")
    plt.ylabel("Manheim Index")
    plt.ylim(0, 370)
    plt.legend(loc="upper left", frameon=False, fontsize=10)

    plt.savefig(f"arima_pid_lm_{i}.png", dpi=300, bbox_inches='tight')
    plt.show()

    plt.fill_between(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),preds[window_length:]- res_decay['q'][window_length+T_burnin:],preds[window_length:]+ res_decay['q'][window_length+T_burnin:], label="OCID Interval", color="C3",alpha=0.2)
    plt.plot(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),preds[window_length:], label ='Predicted Value', color = 'C6')
    plt.plot(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),truths[window_length:], label = 'True Value', color = 'C5')
    plt.xlabel("Date")
    plt.ylabel("Manheim Index")
    plt.ylim(0, 370)
    plt.legend(loc="upper left", frameon=False, fontsize=10)

    plt.savefig(f"arima_ocid_lm_{i}.png", dpi=300, bbox_inches='tight')
    plt.show()

    plt.fill_between(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),preds[window_length:]- res_eci['q'][window_length+T_burnin:],preds[window_length:]+ res_eci['q'][window_length+T_burnin:], label="ECI Interval", color="C4",alpha=0.2)
    plt.plot(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),preds[window_length:], label ='Predicted Value', color = 'C6')
    plt.plot(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),truths[window_length:], label = 'True Value', color = 'C5')
    plt.xlabel("Date")
    plt.ylabel("Manheim Index")
    plt.ylim(0, 370)
    plt.legend(loc="upper left", frameon=False, fontsize=10)

    plt.savefig(f"arima_eci_lm_{i}.png", dpi=300, bbox_inches='tight')
    plt.show()

    plt.fill_between(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),preds[window_length:]- res_dtaci['q'][window_length+T_burnin:],preds[window_length:]+ res_dtaci['q'][window_length+T_burnin:], label="DtACI Interval", color="C5",alpha=0.2)
    plt.plot(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),preds[window_length:], label ='Predicted Value', color = 'C6')
    plt.plot(manheim_df.index[test_last_index-121:test_last_index].to_numpy(),truths[window_length:], label = 'True Value', color = 'C5')
    plt.xlabel("Date")
    plt.ylabel("Manheim Index")
    plt.ylim(0, 370)
    plt.legend(loc="upper left", frameon=False, fontsize=10)

    plt.savefig(f"arima_dtaci_lm_{i}.png", dpi=300, bbox_inches='tight')
    plt.show()